
# 1. **Preprocessing_and_trajectory_analysis**:
 
File organisation

- 0. preprocessing (normalisation, PCA, batch correction) and
- 1. main trajectory inference and analysis.


**Validation:** The following subfolders include the scripts used for validation of the preprocessing and **trajectory inference** part of the analysis. 

**a) Linking_disease_state_to_phenotypes**: Script producing the plots showing agreement of progressing disease characteristics with trajectory position, 

**b) Sex_Differences**: Script producing the correlation plots (disease and general population characteristics against the first 10 principal components) separately for males and females, 

**c) Error_bars_in_histology_per_SW**: Script producing the plots of the average histological scores per SW (including the error bars in each SW), 

**d) Correlation_analysis_normalised_vs_batchcorrected_counts**: Script producing all the supplementary plots validating successful batch effect correction, including correlations per disease stage and stratified correlation for all disease stages.


## 0. normalisation, PCA, batch correction

*Why?*

merged_counts.csv comes from a prestep bringing data from FASTA and others (documented in methods_for_data.ipynb) . so we created a script mergedcounts_generation.py to autogenerate this. 

you need this merged counts to perform a nroamlizatipn. why a normalization of the counts?

### Why normalization, and why PCA + scree plot? (grounded in this cell's code)

**Why normalize `merged_counts` before anything else:**

```r
boxplot(log2(1+merged_counts), main="Un-Normalized Counts")
nrm=normalize.quantiles(log2(1+merged_counts))
boxplot((nrm), main="Normalized Counts")
```

- `log2(1+x)`: RNA-seq counts are highly skewed (a handful of genes dominate the raw scale) and their variance grows with their mean. Log-transforming compresses that dynamic range so no single gene dominates downstream distance/variance calculations; `+1` avoids `log(0)` for genes with zero counts.
- `normalize.quantiles()` (quantile normalization): forces every sample (column) onto the *same* distribution of values. Before this, each sample's overall count distribution differs because of technical factors — sequencing depth/library size, RNA composition — and because UCAM and VCU/Sanyal are two independent studies. Quantile normalization removes that per-sample technical offset so that any remaining difference in a gene's value *between* samples reflects biology, not depth.
- The script literally draws this as a before/after check — two boxplots side by side. Before normalization the boxes sit at different heights/spreads per sample; after, they're forced to be visually identical (that's what quantile normalization guarantees by construction). This diagnostic is what later becomes part of Supplementary Fig. 1 (see `methods_for_data.ipynb`).
- Why this matters for the rest of the pipeline: the whole point of the paper is to find a *biological* pseudotemporal trajectory (PC1) across the merged UCAM+VCU samples. If depth/technical differences aren't normalized out first, PC1 would end up separating samples by "how deeply was this one sequenced" rather than by disease stage.

**Why PCA, and why a scree plot, right after:**

```r
pca = prcomp(t(nrm))
pca.var <- pca$sdev^2
pca.var.per <- round(pca.var/sum(pca.var)*100, 1)
plot(pca$x[,1], pca$x[,2], col=condition_, ...)   # PC1 vs PC2, coloured by CTRL/MASL/MASH F0-2/MASH F3-4
barplot(pca.var.per, main="Scree Plot", ...)
```

- `prcomp(t(nrm))` reduces the ~17,090-gene expression matrix to a small number of orthogonal axes (principal components) ranked by how much cross-sample variance they capture. This is the exploratory check for the paper's central claim: that MASLD progression can be captured by a single dominant molecular axis. Colouring PC1/PC2 by disease stage (CTRL → MASL → MASH F0-2 → MASH F3-4) is a visual test of whether that axis already separates patients by severity *before* any trajectory-inference method (Slingshot) is even applied.
- The scree plot (`pca.var.per` as a barplot) shows how much variance each successive PC explains. A steep drop after PC1 is what justifies treating PC1 alone as *the* pseudotemporal trajectory axis later in the pipeline, rather than needing a multi-dimensional embedding — it's the evidence that one component concentrates most of the meaningful signal. This is the same `pca.var.per[1]` number the paper's Methods (p.11) cites when describing how much variance PC1 explains pre- vs. post- full batch correction.
- Note this PCA runs on the quantile-normalized counts *before* ComBat batch correction (that happens in the separate "Batch correction UCAM-VCU..." script) — so this specific plot is a QC step confirming the biological signal is visible even before removing the UCAM/VCU batch effect, not yet the final trajectory.


In [ ]:
# NORMALIZATION & PCA

#Normalization, PCA and Screeplot
#Ucam & VCU (Sanyal) datasets (human MASLD datasets)

library("preprocessCore")
library(readxl)

#Convert counts to matrix and exlude low expressed counts
merged_counts = read.csv(file = "../data/mergedcounts.csv", sep = ",")
rownames(merged_counts) = merged_counts$X
merged_counts = as.matrix(merged_counts[,-1])

#Normalization
pdf(file="Human_Boxplot of counts.pdf")
par(mfrow=c(1,2), mar = c(4,3,2,4))
boxplot(log2(1+merged_counts), main="Un-Normalized Counts")
nrm=normalize.quantiles(log2(1+merged_counts))
dimnames(nrm) = dimnames(merged_counts)
boxplot((nrm), main="Normalized Counts")
write.csv(nrm, file ="Human_Normalised_counts.csv")
dev.off()

#Normalization, PCA and Screeplot
template = read.csv(file = "../data/metadata.csv", header = TRUE)[,-1]

unique(template$Sample.name == colnames(nrm))
CTRL = which(template$SAF.score == "CTRL")
MASL = which(template$SAF.score == "MASL")#red
MASH_F012 = which(template$SAF.score == "MASH F0" | template$SAF.score == "MASH F1" | template$SAF.score == "MASH F2") #lightblue
MASH_F34 = which(template$SAF.score == "MASH F3" | template$SAF.score == "MASH F4") #black

#PCA plot
#Points for the different stages of the PCA plot
pca = prcomp(t(nrm))
pca.var <- pca$sdev^2
pca.var.per <- round(pca.var/sum(pca.var)*100, 1) #How much variation in the original data each PC accounts for

condition_ = rep("#de2d26", dim(nrm)[2]) #MASH_F34
condition_[MASH_F012]="#fb6a4a" #MASH_F0123
condition_[MASL]="#fcae91" #MASL
condition_[CTRL]="#3182bd" #CTRL

pdf(file="Human_UCAM&VCU_PCA.pdf")
plot(pca$x[,1], pca$x[,2], col=condition_, xlab=paste("PC1"," - ", pca.var.per[1],"%"), ylab=paste("PC2"," - ", pca.var.per[2],"%"), main = "Normalised Counts\n (Merged UCAM & VCU dataset)", pch=16, cex = 1.2)
legend(x="top", legend = c("CTRL", "MASL","MASHF012","MASHF34"), col=c("#3182bd", "#fcae91", "#fb6a4a", "#de2d26"), pch=16, cex=1.2, box.lty = 0)
dev.off()

pdf(file="Human_ScreePlot.pdf")
barplot(pca.var.per, main="Scree Plot", xlab="Principal Components", ylab="Percent Variation", names.arg = pca.var.per)
dev.off()






In [ ]:
# BATCH CORRECTION



library("preprocessCore")
#install.packages("plotly")
library(plotly)
library(tidyverse)
library("sva")
library(readxl)

#merged Ucam/VCU counts
merged_counts <- read.csv("../data/mergedcounts.csv")
rownames(merged_counts) = merged_counts$X
merged_counts = merged_counts[,-1]

#remove outlier - Sample 5
merged_counts = merged_counts[ , -5]

#merged template
template <- read.csv("../data/metadata.csv")[,-1]
template = template[-5,] #remove outlier
colnames(merged_counts) = template$Sample.name

#Convert counts to matrix and exlude low expressed counts
merged_counts = data.matrix(merged_counts)
merged_counts <- merged_counts[(rowSums(merged_counts)>dim(merged_counts)[2]),]

#Normalization
nrm=normalize.quantiles(log2(1+merged_counts))
dimnames(nrm) = dimnames(merged_counts)

batch_ = paste(template$Dataset, template$Sex)
mod_ = model.matrix(~as.factor(template$NAS))
corrected <- ComBat(dat=as.matrix(nrm), batch=batch_, mod=mod_, par.prior=TRUE, prior.plots=FALSE)
write.csv(corrected, file = "batch_corrected_counts_(dataset+gender).csv")

unique(template$Sample.name == colnames(corrected))

CTRL = which(template$SAF.score == "CTRL")
MASL = which(template$SAF.score == "MASL")#red
MASH_F012 = which(template$SAF.score == "MASH F0" | template$SAF.score == "MASH F1" | template$SAF.score == "MASH F2") #lightblue
MASH_F34 = which(template$SAF.score == "MASH F3" | template$SAF.score == "MASH F4") #black



#PCA plot --- After batch correction
pca = prcomp(t(corrected))
pca.var <- pca$sdev^2
pca.var.per <- round(pca.var/sum(pca.var)*100, 1) #How much variation in the original data each PC accounts for

condition_ = rep("#de2d26", dim(corrected)[2]) #MASH_F34
condition_[MASH_F012]="#fb6a4a" #MASH_F0123
condition_[MASL]="#fcae91" #MASL
condition_[CTRL]="#3182bd" #CTRL

pdf(file="Human_UCAM&VCU_PCA_afterbatchcorrection_excludingoutlier.pdf")
plot(pca$x[,1], pca$x[,2], col=condition_, xlab=paste("PC1"," - ", pca.var.per[1],"%"), ylab=paste("PC2"," - ", pca.var.per[2],"%"), main = "Batch Corrected Counts\n (Merged UCAM & VCU dataset - excluding outlier)", pch=16, cex = 1.2)
legend(x="topleft", legend = c("CTRL", "MASL","MASHF012","MASHF34"), col=c("#3182bd", "#fcae91", "#fb6a4a", "#de2d26"), pch=16, cex=1.2, box.lty = 0)
dev.off()

pdf(file="Human_ScreePlot.pdf")
barplot(pca.var.per, main="Scree Plot", xlab="Principal Components", ylab="Percent Variation", names.arg = pca.var.per)
dev.off()






In [ ]:
# SLINGSHOT UCAM&VCU after BATCH CORRECTION

##Analysis was based on the example of this Vignette:
#https://www.bioconductor.org/packages/release/bioc/vignettes/slingshot/inst/doc/vignette.html

# if (!requireNamespace("BiocManager", quietly = TRUE))
#   install.packages("BiocManager")
# 
# BiocManager::install("slingshot")

library("slingshot")
library("SingleCellExperiment")

counts=read.csv("../data/batch_corrected_counts_(dataset+gender).csv", sep = ",")
rownames(counts) = counts$X
counts = counts[ ,-c(1)]
counts = round(counts)
template=read.csv(file = "../data/metadata.csv",header = TRUE ,stringsAsFactors = FALSE,row.names = 1)
rownames(template) = template$Sample.name
colnames(counts) = template$Sample_names
sim <- SingleCellExperiment(assays = List(counts = as.matrix(counts)))

#Dimensionality Reduction
pca <- prcomp(t(assays(sim)$counts), scale. = FALSE)
pca.var <- pca$sdev^2
pca.var.per <- round(pca.var/sum(pca.var)*100, 1) #How much variation in the original data each PC accounts for
rd1 <- pca$x[,1:2]

rescale <- function(rd1)
{
  max_ = max(abs(rd1[,2]))
  for (i in 1:length(rd1[,2]))
  {
    if(abs(rd1[i,2]) > max_/3) rd1[i,2] = rd1[i,2] * .2
    else if(abs(rd1[i,2]) > max_/5) rd1[i,2] = rd1[i,2] * .4
    else if(abs(rd1[i,2]) > max_/10) rd1[i,2] = rd1[i,2] * .6
  }
  return(rd1)
}

rd1 = rescale(rd1)
reducedDims(sim) <- SimpleList(PCA = rd1)

#Clustering Cells
library(mclust, quietly = TRUE)
cl1 <- Mclust(rd1)$classification
colData(sim)$GMM <- cl1

library(RColorBrewer)
#plot(rd1, col = brewer.pal(9,"Set1")[cl1], pch=16, asp = 1)

cl2 <- kmeans(rd1, centers = 7)$cluster
colData(sim)$kmeans <- cl2
#plot(rd1, col = brewer.pal(9,"Set1")[cl2], pch=16, asp = 1)


#Using Slingshot
#Uses either "GMM" or "kmeans" - kmeans will have a number of clusters = centers
sim <- slingshot(sim, clusterLabels = 'GMM', reducedDim = 'PCA')
#sim <- slingshot(sim, clusterLabels = 'kmeans', reducedDim = 'PCA')

#change colors for plot
col_ = template$NAS
col_[template$NAS ==0] = "#081d58"
col_[template$NAS ==1] = "#253494"
col_[template$NAS ==2] = "#225ea8"
col_[template$NAS ==3] = "#1d91c0"
col_[template$NAS ==4] = "#41b6c4"
col_[template$NAS ==5] = "#7fcdbb"
col_[template$NAS ==6] = "#c7e9b4"
col_[template$NAS ==7] = "#edf8b1"
col_[template$NAS ==8] = "#ffffd9"

pdf(file="Pseudo_UCAM&VCU_aftercorrection.pdf")
plot(reducedDims(sim)$PCA[,1], reducedDims(sim)$PCA[,2] * (pca.var.per[2] / pca.var.per[1]), col = col_, pch=16, cex = .6,
     ylim=c( -max(abs(reducedDims(sim)$PCA[,1])) - 20, max(abs(pca$x[,1])) +10),
     main = "Pseudotemporal Ordering \n UCAM/VCU (After batch correction)", xlab="Pseudotime", ylab=" ")
lines(SlingshotDataSet(sim), lwd=2, col='black')
legend("top", legend=rev(c("0", "1", "2", "3", "4", "5", "6", "7")), horiz = T,
       col=rev(c("#081d58","#253494","#225ea8","#1d91c0","#41b6c4","#7fcdbb","#c7e9b4","#edf8b1")), pch=20, cex=0.78, bty = "n")
dev.off()





## 1.a.Linking_disease_state_to_phenotypes

Script producing the plots showing agreement of progressing disease characteristics with trajectory position



In [ ]:
# Load data
template <- read.csv(file = "data/metadata.csv",
                     header = TRUE, stringsAsFactors = FALSE, row.names = 1)
template <- template[template$Sample.name != "Sample 5", ]

sorted_samples <- read.csv("data/PC1_sorted_samples.csv")

# Merge on Sample name
merged_data <- merge(template, sorted_samples, by.x = "Sample.name", by.y = "X")
merged_data$PC1 <- -merged_data$PC1

# Get only the relevant columns
df <- merged_data[, c("Sample.name", "PC1", "NAS")]

# Generate all pairwise combinations
combinations <- expand.grid(1:nrow(df), 1:nrow(df))
combinations <- combinations[combinations$Var1 < combinations$Var2, ]  # Remove self-pairs and duplicates

# Compute pairwise differences
pairwise_df <- data.frame(
  Sample1 = df$Sample.name[combinations$Var1],
  Sample2 = df$Sample.name[combinations$Var2],
  PC1_diff = abs(df$PC1[combinations$Var1] - df$PC1[combinations$Var2]),
  NAS_diff = abs(df$NAS[combinations$Var1] - df$NAS[combinations$Var2])
)

library(ggplot2)
library(gridExtra)

# Function to make individual plots
make_plot <- function(y, y_label) {
  cor_res <- cor.test(pairwise_df$PC1_diff, pairwise_df[[y]], method = "pearson")
  
  # Format p-value for very small values
  p_value_formatted <- ifelse(cor_res$p.value < 1e-300, "< 1e-300", 
                              formatC(cor_res$p.value, format = "e", digits = 2))
  
  ggplot(pairwise_df, aes(x = PC1_diff, y = !!as.name(y))) +
    geom_point(alpha = 0.3, color = "darkblue") +
    geom_smooth(method = "lm", se = TRUE, color = "orange") +
    labs(
      title = paste0(y_label, "\nR = ", round(cor_res$estimate, 2),
                     ", p = ", p_value_formatted),
      x = "Pairwise difference on trajectory",
      y = paste0(y_label, " Difference")
    ) +
    theme_minimal(base_size = 12)
}

# Compute pairwise differences for all histology scores
# Signed differences (Var2 - Var1)
pairwise_df$PC1_diff <- merged_data$PC1[combinations$Var2] - merged_data$PC1[combinations$Var1]
pairwise_df$NAS_diff <- merged_data$NAS[combinations$Var2] - merged_data$NAS[combinations$Var1]
pairwise_df$Steatosis_diff <- merged_data$Fat[combinations$Var2] - merged_data$Fat[combinations$Var1]
pairwise_df$Inflammation_diff <- merged_data$Inflammation[combinations$Var2] - merged_data$Inflammation[combinations$Var1]
pairwise_df$Ballooning_diff <- merged_data$Ballooning[combinations$Var2] - merged_data$Ballooning[combinations$Var1]
pairwise_df$Fibrosis_diff <- merged_data$Fibrosis[combinations$Var2] - merged_data$Fibrosis[combinations$Var1]

# Create all plots
p1 <- make_plot("NAS_diff", "NAS")
p2 <- make_plot("Steatosis_diff", "Steatosis")
p3 <- make_plot("Inflammation_diff", "Inflammation")
p4 <- make_plot("Ballooning_diff", "Ballooning")
p5 <- make_plot("Fibrosis_diff", "Fibrosis")

# Combine into a multi-panel figure (2 rows)
grid.arrange(p1, p2, p3, p4, p5, ncol = 2)


## 1.b. Sex_Differences

Script producing the correlation plots (disease and general population characteristics against the first 10 principal components) separately for males and females


In [ ]:
# FEMALES



library("preprocessCore")
#install.packages("plotly")
library(tidyverse)
library("sva")

library(readxl)


#merged Ucam/Sanyal
merged_counts <- read.csv("data/mergedcounts.csv", check.names = F)
rownames(merged_counts) = merged_counts[,1]
merged_counts = merged_counts[,-1]

#merged template
template <- read.csv("data/metadata.csv")[,-1]
table(is.element(colnames(merged_counts), template$Sample.name)) #when reading the csv file, it put a "." in the sample names that created confusion - I changed the names to agree!



#Separate in males/females
counts_females = merged_counts[ , template$Sex == "F"]
template_females = template[ template$Sex == "F", ]
rm(merged_counts, template)

#Convert counts to matrix and exclude low expressed counts
counts_females = data.matrix(counts_females)
counts_females <- counts_females[(rowSums(counts_females)>dim(counts_females)[2]),]

#Normalization
nrm_females=normalize.quantiles(log2(1+counts_females))
dimnames(nrm_females) = dimnames(counts_females)



mod_females = model.matrix(~as.factor(template_females$NAS))
corrected_counts_females <- ComBat(dat=as.matrix(nrm_females), batch=template_females$Dataset, mod=mod_females, par.prior=TRUE, prior.plots=FALSE)
write.csv(corrected_counts_females, file = "Batch Corrected Counts (UCAM-Sanyal) - with mod_NAS_only females.csv")

rm(mod_females, counts_females, nrm_females)




# 
# #PCA Plots - Version1
# #####
pca = prcomp(t(corrected_counts_females))
pca.var <- pca$sdev^2
pca.var.per <- round(pca.var/sum(pca.var)*100, 1) #How much variation in the original data each PC accounts for


 
 
 
 
CTRL = which(template_females$SAF.score == "CTRL")
MASL = which(template_females$SAF.score == "MASL")#red
MASH_F012 = which(template_females$SAF.score == "MASH F0" | template_females$SAF.score == "MASH F1" | template_females$SAF.score == "MASH F2") #lightblue
MASH_F34 = which(template_females$SAF.score == "MASH F3" | template_females$SAF.score == "MASH F4") #black
 
 
condition_ = rep("#de2d26", dim(corrected_counts_females)[2]) #MASH_F34
condition_[MASH_F012]="#fb6a4a" #MASH_F0123
condition_[MASL]="#fcae91" #MASL
condition_[CTRL]="#3182bd" #CTRL
 
 

pdf(file="Human_UCAM&SANYAL_PCA_females.pdf")
plot(pca$x[,1], pca$x[,2], col=condition_, xlab=paste("PC1"," - ", pca.var.per[1],"%"), ylab=paste("PC2"," - ", pca.var.per[2],"%"), main = "Normalised Counts\n (Merged UCAM & Sanyal dataset)", pch=16, cex = 1.2)
legend(x="topright", legend = c("CTRL", "MASL","MASHF012","MASHF34"), col=c("#3182bd", "#fcae91", "#fb6a4a", "#de2d26"), pch=16, cex=1.2, box.lty = 0)
#text(pca$x[,1], pca$x[,2], colnames(nrm),pos = 1, offset=.1,cex=.2)
dev.off()
 

 
 
 
 



# Find confounding factors for all the batch_corrected counts together
###Part1
# First, I did it for the continuous variables, for "BW", "Liver", "LW.BW.", "Glucose", "ALT", "Insulin", 'TGs', "Cholesterol", "HDL", "LDL", "AST", "ALP", "Hepatic.Cholesterol", "Hepatic.TGs" , "Daily.Food.Intake..average."
# I correlated and created heatmaps using pvalues or cc (correlation value between 0 and 1)
# These are variables that (hopefully) could be correlated with the different PCs (Principal Components)
# I repeated the test for 4 different combinations (1.batch_corrected + phenotypes_without_log2_transformation, 2.batch_corrected + log2_transformed_(phenotypes), 3.normalised + phenotypes_without_log2_transformation), 4.normalised + log2_transformed_(phenotypes)
# Finally, I compare the non corrected vs the corrected  - for the log2 transformed - to verify that the biology is responsible for the variation in the principle components separation
###Part2
# I repeated the same (only the pvalues using anova) for the cotegorical values ("Partner","Diet.Group","Type.of.Cage","Sugar.Water.","Diet.started.at.age..week.","Duration.Experimental.Diet..weeks.")
# In that case I didn't use the log of the codes, cause they are categorial values
library("preprocessCore")


corrected <- read.csv(file = "Batch Corrected Counts (UCAM-Sanyal) - with mod_NAS_only females.csv", sep = ',', header = TRUE, check.names = F)
rownames(corrected) = corrected[,1]
corrected = corrected[,-1]

codes <- template_females
colnames(corrected) = codes$Sample.name
unique(colnames(corrected) == codes$Sample.name)
codes = codes[,-3] #remove the sex variable

for (j in 1:dim(codes)[2]) 
{
  print(colnames(codes)[j])
  print(summary(codes[,j]))
}


#For the continuous variables
codes = codes[ , c( "Age", "NAS", "Fibrosis", "Fat", "Ballooning", "Inflammation")]
#codes = log2(codes) #### ### ### ### ### ### To perform a parametric test, the distribution has to be Normal. A way to achive that sometimes is via log-transformation


#PCA Plots
pca = prcomp(t(corrected))

confactors= correlation = matrix(0, nrow = 10, ncol = dim(codes)[2])
for (i in 1:10)
  for (j in 1:dim(codes)[2])
  {
    a = cor.test(pca$x[,i], codes[,j], method = "pearson")
    correlation[i,j] = a$estimate
    confactors[i,j] = a$p.value
    #confactors[i,j] = - log10(a$p.value +.01)
  }
rownames(confactors) =  rownames(correlation) = c("PC1","PC2","PC3","PC4","PC5","PC6","PC7","PC8","PC9","PC10")
colnames(confactors) = colnames(correlation)  =colnames(codes)




colored_pvals = confactors

library(pheatmap)
paletteLength <- 21
myColor <- colorRampPalette(c( "red","white"))(paletteLength)
myBreaks <- seq(0, 1, by = .01)
pheatmap(colored_pvals, color=myColor, cluster_rows = FALSE, breaks=myBreaks, display_numbers = round(confactors,3), cluster_cols=FALSE, fontsize_row = 10, fontsize_col = 10, main = "P-value for continuous variables\n(mod = NAS) - Females", file = "P-vals_females.png")

myColor <- colorRampPalette(c( "green","white", "red"))(paletteLength)
myBreaks <- seq(-1, 1, by = .1)
pheatmap(correlation, color=myColor, cluster_rows = FALSE, breaks=myBreaks, display_numbers = round(correlation,3), cluster_cols=FALSE, fontsize_row = 10, fontsize_col = 10, main = "Correlation Plot for continuous variables\n(mod = NAS) - Females", file = "Correlation_females.png")


#I performed the Shapiro-Wilk's method to test for Normality. If the p-value<.05 it is assumed that the distribution is not normal

for (i in 1:dim(codes)[2])
{
  print(colnames(codes)[i])
  test = shapiro.test(codes[,i])
  print(test$p.value)
  if(test$p.value >.05)
    print("Normal Distribution")
  else
    print("Non-Normality")
}










###For the categorical variable
codes = template_females[ , c("Age","NAS","T2DM","Dataset")]
allcodes =template_females
#
codes$Age[allcodes$Age<45] <- "young"
codes$Age[allcodes$Age>=45 & codes$Age<60] <- "middle"
codes$Age[allcodes$Age>=60] <- "old"


for (j in 1:dim(codes)[2]) 
{
  print(colnames(codes)[j])
  print(summary(codes[,j]))
  print(typeof(codes[,j]))
}


#PCA Plots
pca = prcomp(t(corrected))

confactors = matrix(2, nrow = 10, ncol = dim(codes)[2]) #The matrix contains the pvals
for (i in 1:10)
  for (j in 1:dim(codes)[2])
  {
    df = data.frame(codes[,j], pca$x[,i])
    age.aov = aov(pca$x[,i]~codes[,j], data=df)
    s = unlist(summary(age.aov))
    confactors[i,j] = s[9]
  }
rownames(confactors) = c("PC1","PC2","PC3","PC4","PC5","PC6","PC7","PC8","PC9","PC10")
colnames(confactors) =colnames(codes)

library(pheatmap)
paletteLength <- 21
myColor <- colorRampPalette(c( "red","white"))(paletteLength)
myBreaks <- seq(0, 1, by = .01)
pheatmap(confactors, color=myColor, cluster_rows = FALSE, breaks=myBreaks, display_numbers = round(confactors,5), cluster_cols=FALSE, fontsize_row = 10, fontsize_col = 10, main = "P-value for categorical variables\n(mod = NAS) - Females", file = "P-vals_categorical variables_females.png")







In [ ]:
# MALES



library("preprocessCore")
#install.packages("plotly")
library(tidyverse)
library("sva")

library(readxl)


#merged Ucam/Sanyal
merged_counts <- read.csv("data/mergedcounts.csv", check.names = F)
rownames(merged_counts) = merged_counts[,1]
merged_counts = merged_counts[,-1]

#merged template
template <- read.csv("data/metadata.csv")[,-1]
table(is.element(colnames(merged_counts), template$Sample.name)) #when reading the csv file, it put a "." in the sample names that created confusion - I changed the names to agree!
template = template[ -which(is.element(template$Sample.name, c("Sample 41", "Sample 43"))), ]
merged_counts = merged_counts[ , template$Sample.name]

#Separate in males/females
counts_males = merged_counts[ , template$Sex == "M"]
template_males = template[ template$Sex == "M", ]
rm(merged_counts, template)

#Convert counts to matrix and exclude low expressed counts
counts_males = data.matrix(counts_males)
counts_males <- counts_males[(rowSums(counts_males)>dim(counts_males)[2]),]

#Normalization
nrm_males=normalize.quantiles(log2(1+counts_males))
dimnames(nrm_males) = dimnames(counts_males)



mod_males = model.matrix(~as.factor(template_males$NAS))
corrected_counts_males <- ComBat(dat=as.matrix(nrm_males), batch=template_males$Dataset, mod=mod_males, par.prior=TRUE, prior.plots=FALSE)
write.csv(corrected_counts_males, file = "Batch Corrected Counts (UCAM-Sanyal) - with mod_NAS_only males.csv")

rm(mod_males, counts_males, nrm_males)




# 
# #PCA Plots - Version1
# #####
pca = prcomp(t(corrected_counts_males))
pca.var <- pca$sdev^2
pca.var.per <- round(pca.var/sum(pca.var)*100, 1) #How much variation in the original data each PC accounts for


 
 
 
 
CTRL = which(template_males$SAF.score == "CTRL")
MASL = which(template_males$SAF.score == "MASL")#red
MASH_F012 = which(template_males$SAF.score == "MASH F0" | template_males$SAF.score == "MASH F1" | template_males$SAF.score == "MASH F2") #lightblue
MASH_F34 = which(template_males$SAF.score == "MASH F3" | template_males$SAF.score == "MASH F4") #black
 
 
condition_ = rep("#de2d26", dim(corrected_counts_males)[2]) #MASH_F34
condition_[MASH_F012]="#fb6a4a" #MASH_F0123
condition_[MASL]="#fcae91" #MASL
condition_[CTRL]="#3182bd" #CTRL
 
 

pdf(file="Human_UCAM&SANYAL_PCA_males.pdf")
plot(pca$x[,1], pca$x[,2], col=condition_, xlab=paste("PC1"," - ", pca.var.per[1],"%"), ylab=paste("PC2"," - ", pca.var.per[2],"%"), main = "Normalised Counts\n (Merged UCAM & Sanyal dataset)", pch=16, cex = 1.2)
legend(x="bottomright", legend = c("CTRL", "MASL","MASHF012","MASHF34"), col=c("#3182bd", "#fcae91", "#fb6a4a", "#de2d26"), pch=16, cex=1.2, box.lty = 0)
#text(pca$x[,1], pca$x[,2], colnames(nrm),pos = 1, offset=.1,cex=.2)
dev.off()
 

 
 
 
 



# Find confounding factors for all the batch_corrected counts together
###Part1
# First, I did it for the continuous variables, for "BW", "Liver", "LW.BW.", "Glucose", "ALT", "Insulin", 'TGs', "Cholesterol", "HDL", "LDL", "AST", "ALP", "Hepatic.Cholesterol", "Hepatic.TGs" , "Daily.Food.Intake..average."
# I correlated and created heatmaps using pvalues or cc (correlation value between 0 and 1)
# These are variables that (hopefully) could be correlated with the different PCs (Principal Components)
# I repeated the test for 4 different combinations (1.batch_corrected + phenotypes_without_log2_transformation, 2.batch_corrected + log2_transformed_(phenotypes), 3.normalised + phenotypes_without_log2_transformation), 4.normalised + log2_transformed_(phenotypes)
# Finally, I compare the non corrected vs the corrected  - for the log2 transformed - to verify that the biology is responsible for the variation in the principle components separation
###Part2
# I repeated the same (only the pvalues using anova) for the cotegorical values ("Partner","Diet.Group","Type.of.Cage","Sugar.Water.","Diet.started.at.age..week.","Duration.Experimental.Diet..weeks.")
# In that case I didn't use the log of the codes, cause they are categorial values
library("preprocessCore")


corrected <- read.csv(file = "Batch Corrected Counts (UCAM-Sanyal) - with mod_NAS_only males.csv", sep = ',', header = TRUE, check.names = F)
rownames(corrected) = corrected[,1]
corrected = corrected[,-1]

codes <- template_males
colnames(corrected) = codes$Sample.name
unique(colnames(corrected) == codes$Sample.name)
codes = codes[,-3] #remove the sex variable

for (j in 1:dim(codes)[2]) 
{
  print(colnames(codes)[j])
  print(summary(codes[,j]))
}


#For the continuous variables
codes = codes[ , c( "Age", "NAS", "Fibrosis", "Fat", "Ballooning", "Inflammation")]
#codes = log2(codes) #### ### ### ### ### ### To perform a parametric test, the distribution has to be Normal. A way to achive that sometimes is via log-transformation


#PCA Plots
pca = prcomp(t(corrected))

confactors= correlation = matrix(0, nrow = 10, ncol = dim(codes)[2])
for (i in 1:10)
  for (j in 1:dim(codes)[2])
  {
    a = cor.test(pca$x[,i], codes[,j], method = "pearson")
    correlation[i,j] = a$estimate
    confactors[i,j] = a$p.value
    #confactors[i,j] = - log10(a$p.value +.01)
  }
rownames(confactors) =  rownames(correlation) = c("PC1","PC2","PC3","PC4","PC5","PC6","PC7","PC8","PC9","PC10")
colnames(confactors) = colnames(correlation)  =colnames(codes)




colored_pvals = confactors

library(pheatmap)
paletteLength <- 21
myColor <- colorRampPalette(c( "red","white"))(paletteLength)
myBreaks <- seq(0, 1, by = .01)
pheatmap(colored_pvals, color=myColor, cluster_rows = FALSE, breaks=myBreaks, display_numbers = round(confactors,3), cluster_cols=FALSE, fontsize_row = 10, fontsize_col = 10, main = "P-value for continuous variables\n(mod = NAS) - Males", file = "P-vals_males.png")

myColor <- colorRampPalette(c( "green","white", "red"))(paletteLength)
myBreaks <- seq(-1, 1, by = .1)
pheatmap(correlation, color=myColor, cluster_rows = FALSE, breaks=myBreaks, display_numbers = round(correlation,3), cluster_cols=FALSE, fontsize_row = 10, fontsize_col = 10, main = "Correlation Plot for continuous variables\n(mod = NAS) - Males", file = "Correlation_males.png")


#I performed the Shapiro-Wilk's method to test for Normality. If the p-value<.05 it is assumed that the distribution is not normal

for (i in 1:dim(codes)[2])
{
  print(colnames(codes)[i])
  test = shapiro.test(codes[,i])
  print(test$p.value)
  if(test$p.value >.05)
    print("Normal Distribution")
  else
    print("Non-Normality")
}










###For the categorical variable
codes = template_males[ , c("Age","NAS","T2DM","Dataset")]
allcodes =template_males

#
codes$Age[allcodes$Age<45] <- "young"
codes$Age[allcodes$Age>=45 & codes$Age<60] <- "middle"
codes$Age[allcodes$Age>=60] <- "old"


for (j in 1:dim(codes)[2]) 
{
  print(colnames(codes)[j])
  print(summary(codes[,j]))
  print(typeof(codes[,j]))
}


#PCA Plots
pca = prcomp(t(corrected))

confactors = matrix(2, nrow = 10, ncol = dim(codes)[2]) #The matrix contains the pvals
for (i in 1:10)
  for (j in 1:dim(codes)[2])
  {
    df = data.frame(codes[,j], pca$x[,i])
    age.aov = aov(pca$x[,i]~codes[,j], data=df)
    s = unlist(summary(age.aov))
    confactors[i,j] = s[9]
  }
rownames(confactors) = c("PC1","PC2","PC3","PC4","PC5","PC6","PC7","PC8","PC9","PC10")
colnames(confactors) =colnames(codes)

library(pheatmap)
paletteLength <- 21
myColor <- colorRampPalette(c( "red","white"))(paletteLength)
myBreaks <- seq(0, 1, by = .01)
pheatmap(confactors, color=myColor, cluster_rows = FALSE, breaks=myBreaks, display_numbers = round(confactors,5), cluster_cols=FALSE, fontsize_row = 10, fontsize_col = 10, main = "P-value for categorical variables\n(mod = NAS) - Males", file = "P-vals_categorical variables_males.png")







## 1.c. Error_bars_in_histology_per_SW

Script producing the plots of the average histological scores per SW (including the error bars in each SW)


In [ ]:
# Load required scripts and libraries
source("~/Desktop/DESeq2_Analysis_Functions.R")
source("~/Desktop/Plotly_plots.R")
library(readxl)
library("ggpubr")
library("cowplot")

# Load datasets
patients_in_SWs = read.csv("data/Patients in SWs.csv")

template = read.csv(file = "data/metadata.csv")[, -c(1,3)]
cts_ucamsanyal <- read.csv(file = "data/mergedcounts.csv", sep = ",")
rownames(cts_ucamsanyal) = as.character(cts_ucamsanyal$X)
cts_ucamsanyal = cts_ucamsanyal[,-c(1)]
colnames(cts_ucamsanyal) = template$Sample.name
cts_ucamsanyal <- cts_ucamsanyal[(rowSums(cts_ucamsanyal)>dim(cts_ucamsanyal)[2]),] #Exclude low expressed counts

# Initialize vectors
meanNASinSWs = meanSteatosisinSWs = meanInflammationinSWs = meanBallooninginSWs = meanFibrosisinSWs = SW_number = rep(-2, dim(patients_in_SWs)[2])
semNAS = semSteatosis = semInflammation = semBallooning = semFibrosis = rep(NA, dim(patients_in_SWs)[2])

# Compute means and SEMs
for (i in 1:dim(patients_in_SWs)[2]) {
   patients_in_sw = patients_in_SWs[,i]
   patients_in_sw = patients_in_sw[patients_in_sw != ""]
   
   idx = is.element(template$Sample.name, patients_in_sw)
   
   meanNASinSWs[i] = mean(template$NAS[idx])
   semNAS[i] = sd(template$NAS[idx]) / sqrt(sum(idx))
   
   meanSteatosisinSWs[i] = mean(template$Fat[idx])
   semSteatosis[i] = sd(template$Fat[idx]) / sqrt(sum(idx))
   
   meanInflammationinSWs[i] = mean(template$Inflammation[idx])
   semInflammation[i] = sd(template$Inflammation[idx]) / sqrt(sum(idx))
   
   meanBallooninginSWs[i] = mean(template$Ballooning[idx])
   semBallooning[i] = sd(template$Ballooning[idx]) / sqrt(sum(idx))
   
   meanFibrosisinSWs[i] = mean(template$Fibrosis[idx])
   semFibrosis[i] = sd(template$Fibrosis[idx]) / sqrt(sum(idx))
}

# Plot NAS score with error bars
pdf("Average NAS score.pdf", width=8, height=3)
options(repr.plot.width=8, repr.plot.height=2)
range_meanNASinSWs = range(meanNASinSWs + semNAS, meanNASinSWs - semNAS)
range_meanNASinSWs[1] = range_meanNASinSWs[1] - .5
range_meanNASinSWs[2] = range_meanNASinSWs[2] + .5

plot(1, type="n", xlab="", ylab="", xlim=c(1, length(meanNASinSWs)), ylim=range_meanNASinSWs, main="", xaxt="n")

# Add lines and points
lines(meanNASinSWs, type="o", pch=16, col="darkblue", lwd=2)
points(meanNASinSWs, pch=16, col="darkblue", cex=1.5)

# Add error bars
arrows(x0=1:length(meanNASinSWs), y0=meanNASinSWs - semNAS,
       x1=1:length(meanNASinSWs), y1=meanNASinSWs + semNAS,
       angle=90, code=3, length=0.05, col="darkblue", lwd=1.5)

# Customize axis
axis(1, at=1:length(meanNASinSWs), labels=1:length(meanNASinSWs))
mtext("NAS score", side=2, line=2.5)

dev.off()




pdf("Average SBIF with error bars.pdf", width=8, height=3)
# Set up the plotting area
options(repr.plot.width=8, repr.plot.height=2)
plot(1, type="n", xlab="", ylab="", xlim=c(1, length(meanSteatosisinSWs)), ylim=c(0, 4), main="", xaxt="n")

# Add lines and points
lines(meanSteatosisinSWs, type="o", pch=16, col="#a6611a", lwd=2)
points(meanSteatosisinSWs, pch=16, col="#a6611a", cex=1.5)

lines(meanBallooninginSWs, type="o", pch=16, col="#80cdc1", lwd=2)
points(meanBallooninginSWs, pch=16, col="#80cdc1", cex=1.5)

lines(meanInflammationinSWs, type="o", pch=16, col="#dfc27d", lwd=2)
points(meanInflammationinSWs, pch=16, col="#dfc27d", cex=1.5)

lines(meanFibrosisinSWs, type="o", pch=16, col="#018571", lwd=2)
points(meanFibrosisinSWs, pch=16, col="#018571", cex=1.5)

# Add error bars
arrows(x0=1:13, y0=meanSteatosisinSWs - semSteatosis, y1=meanSteatosisinSWs + semSteatosis,
       code=3, angle=90, length=0.05, col="#a6611a")
arrows(x0=1:13, y0=meanBallooninginSWs - semBallooning, y1=meanBallooninginSWs + semBallooning,
       code=3, angle=90, length=0.05, col="#80cdc1")
arrows(x0=1:13, y0=meanInflammationinSWs - semInflammation, y1=meanInflammationinSWs + semInflammation,
       code=3, angle=90, length=0.05, col="#dfc27d")
arrows(x0=1:13, y0=meanFibrosisinSWs - semFibrosis, y1=meanFibrosisinSWs + semFibrosis,
       code=3, angle=90, length=0.05, col="#018571")

# Axes and labels
axis(1, at=1:length(meanNASinSWs), labels=1:length(meanNASinSWs))
mtext("Score", side=2, line=2.5)

# legend("topright", legend=c("Steatosis", "Ballooning", "Inflammation", "Fibrosis"),
#        col=c("#a6611a", "#80cdc1", "#dfc27d", "#018571"), pch=16, lwd=2, cex=0.8)

dev.off()








pdf("Average SBIF 4 panels with error bars.pdf", width=6, height=8)
par(mfrow=c(4,1), mar=c(3, 4, 2, 2))  # 4 rows, 1 column; adjust margins

# Steatosis
plot(meanSteatosisinSWs, type="o", pch=16, col="#a6611a", lwd=2,
     ylab="Steatosis", xlab="", ylim=c(0,4), xaxt="n")
arrows(x0=1:13, y0=meanSteatosisinSWs - semSteatosis, y1=meanSteatosisinSWs + semSteatosis,
       code=3, angle=90, length=0.05, col="#a6611a")
axis(1, at=1:13, labels=1:13)

# Ballooning
plot(meanBallooninginSWs, type="o", pch=16, col="#80cdc1", lwd=2,
     ylab="Ballooning", xlab="", ylim=c(0,4), xaxt="n")
arrows(x0=1:13, y0=meanBallooninginSWs - semBallooning, y1=meanBallooninginSWs + semBallooning,
       code=3, angle=90, length=0.05, col="#80cdc1")
axis(1, at=1:13, labels=1:13)

# Inflammation
plot(meanInflammationinSWs, type="o", pch=16, col="#dfc27d", lwd=2,
     ylab="Inflammation", xlab="", ylim=c(0,4), xaxt="n")
arrows(x0=1:13, y0=meanInflammationinSWs - semInflammation, y1=meanInflammationinSWs + semInflammation,
       code=3, angle=90, length=0.05, col="#dfc27d")
axis(1, at=1:13, labels=1:13)

# Fibrosis
plot(meanFibrosisinSWs, type="o", pch=16, col="#018571", lwd=2,
     ylab="Fibrosis", xlab="Sliding window", ylim=c(0,4))
arrows(x0=1:13, y0=meanFibrosisinSWs - semFibrosis, y1=meanFibrosisinSWs + semFibrosis,
       code=3, angle=90, length=0.05, col="#018571")
axis(1, at=1:13, labels=1:13)

dev.off()







## 1.d. Correlation_analysis_normalised_vs_batchcorrected_counts

Script producing all the supplementary plots validating successful batch effect correction, including correlations per disease stage and stratified correlation for all disease stages.


In [ ]:

corrected = read.csv("data/batch_corrected_counts_(dataset+gender).csv")
rownames(corrected) = corrected$X
corrected = corrected[,-1]
normalised_beforecorrection = read.csv("data/Human_Normalised_counts.csv")
rownames(normalised_beforecorrection) = normalised_beforecorrection$X
normalised_beforecorrection = normalised_beforecorrection[,-1]


colnames(corrected)
colnames(normalised_beforecorrection)
corrected = corrected[  intersect(rownames(normalised_beforecorrection),rownames(corrected)), intersect(colnames(normalised_beforecorrection),colnames(corrected))]
normalised_beforecorrection = normalised_beforecorrection[ intersect(rownames(normalised_beforecorrection),rownames(corrected)) , intersect(colnames(normalised_beforecorrection),colnames(corrected))]

codes = read.csv('data/metadata.csv')[,-1]
codes = codes[ -5 , ]
codes[,1] = colnames(corrected)

normalised_SANYAL_CTRL = rowMeans(subset(normalised_beforecorrection, select = which(codes$Dataset == "SANYAL" & codes$SAF.score == "CTRL"), na.rm = TRUE))
corrected_SANYAL_CTRL = rowMeans(subset(corrected, select = which(codes$Dataset == "SANYAL" & codes$SAF.score == "CTRL"), na.rm = TRUE))

normalised_SANYAL_MASL = rowMeans(subset(normalised_beforecorrection, select = which(codes$Dataset == "SANYAL" & codes$SAF.score == "MASL"), na.rm = TRUE))
corrected_SANYAL_MASL = rowMeans(subset(corrected, select = which(codes$Dataset == "SANYAL" & codes$SAF.score == "MASL"), na.rm = TRUE))

normalised_SANYAL_MASH_F0 = rowMeans(subset(normalised_beforecorrection, select = which(codes$Dataset == "SANYAL" & codes$SAF.score == "MASH F0"), na.rm = TRUE))
corrected_SANYAL_MASH_F0 = rowMeans(subset(corrected, select = which(codes$Dataset == "SANYAL" & codes$SAF.score == "MASH F0"), na.rm = TRUE))

normalised_SANYAL_MASH_F1 = rowMeans(subset(normalised_beforecorrection, select = which(codes$Dataset == "SANYAL" & codes$SAF.score == "MASH F1"), na.rm = TRUE))
corrected_SANYAL_MASH_F1 = rowMeans(subset(corrected, select = which(codes$Dataset == "SANYAL" & codes$SAF.score == "MASH F1"), na.rm = TRUE))

normalised_SANYAL_MASH_F2 = rowMeans(subset(normalised_beforecorrection, select = which(codes$Dataset == "SANYAL" & codes$SAF.score == "MASH F2"), na.rm = TRUE))
corrected_SANYAL_MASH_F2 = rowMeans(subset(corrected, select = which(codes$Dataset == "SANYAL" & codes$SAF.score == "MASH F2"), na.rm = TRUE))

normalised_SANYAL_MASH_F34 = rowMeans(subset(normalised_beforecorrection, select = which(codes$Dataset == "SANYAL" & (codes$SAF.score == "MASH F3" | codes$SAF.score == "MASH F4")), na.rm = TRUE))
corrected_SANYAL_MASH_F34 = rowMeans(subset(corrected, select = which(codes$Dataset == "SANYAL" & (codes$SAF.score == "MASH F3" | codes$SAF.score == "MASH F4")), na.rm = TRUE))




#plot meanNAS in each SW
library("ggpubr")
library(cowplot)
pdf("before-after correction comparison SANYAL.pdf")
mydata = data.frame("normalised_SANYAL_CTRL" = normalised_SANYAL_CTRL, "corrected_SANYAL_CTRL" = corrected_SANYAL_CTRL)
CTRL_SANYAL <- ggscatter(mydata, x = "normalised_SANYAL_CTRL", y = "corrected_SANYAL_CTRL",
                         color = "black", shape = 20, size = 1, alpha = 0.3, # Points color, shape, size and transparency
                         add = "reg.line",  # Add regressin line
                         add.params = list(color = "black", fill = "lightgray"), conf.int = TRUE, 
                         cor.coef = TRUE, cor.method = "pearson",
                         #ylim = c(1,6),
                         xlab = "Normalised CTRL\n(SANYAL)", ylab = "Corrected CTRL\n(SANYAL)", main = "CTRL SANYAL")

mydata = data.frame("normalised_SANYAL_MASL" = normalised_SANYAL_MASL, "corrected_SANYAL_MASL" = corrected_SANYAL_MASL)
MASL_SANYAL <- ggscatter(mydata, x = "normalised_SANYAL_MASL", y = "corrected_SANYAL_MASL",
                         color = "#f3acac", shape = 20, size = 1, alpha = 0.3, # Points color, shape, size and transparency
                         add = "reg.line",  # Add regressin line
                         add.params = list(color = "black", fill = "lightgray"), conf.int = TRUE, 
                         cor.coef = TRUE, cor.method = "pearson",
                         #ylim = c(1,6),
                         xlab = "Normalised MASL\n(SANYAL)", ylab = "Corrected MASL\n(SANYAL)", main = "MASL SANYAL")

mydata = data.frame("normalised_SANYAL_MASH_F0" = normalised_SANYAL_MASH_F0, "corrected_SANYAL_MASH_F0" = corrected_SANYAL_MASH_F0)
MASH_F0_SANYAL <- ggscatter(mydata, x = "normalised_SANYAL_MASH_F0", y = "corrected_SANYAL_MASH_F0",
                         color = "#ea6564", shape = 20, size = 1, alpha = 0.3, # Points color, shape, size and transparency
                         add = "reg.line",  # Add regressin line
                         add.params = list(color = "black", fill = "lightgray"), conf.int = TRUE, 
                         cor.coef = TRUE, cor.method = "pearson",
                         #ylim = c(1,6),
                         xlab = "Normalised MASHF0\n(SANYAL)", ylab = "Corrected MASHF0\n(SANYAL)", main = "MASHF0 SANYAL")

mydata = data.frame("normalised_SANYAL_MASH_F1" = normalised_SANYAL_MASH_F1, "corrected_SANYAL_MASH_F1" = corrected_SANYAL_MASH_F1)
MASH_F1_SANYAL <- ggscatter(mydata, x = "normalised_SANYAL_MASH_F1", y = "corrected_SANYAL_MASH_F1",
                            color = "#de1e1d", shape = 20, size = 1, alpha = 0.3, # Points color, shape, size and transparency
                            add = "reg.line",  # Add regressin line
                            add.params = list(color = "black", fill = "lightgray"), conf.int = TRUE, 
                            cor.coef = TRUE, cor.method = "pearson",
                            #ylim = c(1,6),
                            xlab = "Normalised MASHF1\n(SANYAL)", ylab = "Corrected MASHF1\n(SANYAL)", main = "MASHF1 SANYAL")

mydata = data.frame("normalised_SANYAL_MASH_F2" = normalised_SANYAL_MASH_F2, "corrected_SANYAL_MASH_F2" = corrected_SANYAL_MASH_F2)
MASH_F2_SANYAL <- ggscatter(mydata, x = "normalised_SANYAL_MASH_F2", y = "corrected_SANYAL_MASH_F2",
                            color = "#971414", shape = 20, size = 1, alpha = 0.3, # Points color, shape, size and transparency
                            add = "reg.line",  # Add regressin line
                            add.params = list(color = "black", fill = "lightgray"), conf.int = TRUE, 
                            cor.coef = TRUE, cor.method = "pearson",
                            #ylim = c(1,6),
                            xlab = "Normalised MASHF2\n(SANYAL)", ylab = "Corrected MASHF2\n(SANYAL)", main = "MASHF2 SANYAL")

mydata = data.frame("normalised_SANYAL_MASH_F34" = normalised_SANYAL_MASH_F34, "corrected_SANYAL_MASH_F34" = corrected_SANYAL_MASH_F34)
MASH_F34_SANYAL <- ggscatter(mydata, x = "normalised_SANYAL_MASH_F34", y = "corrected_SANYAL_MASH_F34",
                            color = "#4f0a0a", shape = 20, size = 1, alpha = 0.3, # Points color, shape, size and transparency
                            add = "reg.line",  # Add regressin line
                            add.params = list(color = "black", fill = "lightgray"), conf.int = TRUE, 
                            cor.coef = TRUE, cor.method = "pearson",
                            #ylim = c(1,6),
                            xlab = "Normalised MASHF34\n(SANYAL)", ylab = "Corrected MASHF34\n(SANYAL)", main = "MASHF34 SANYAL")

plot_grid(CTRL_SANYAL, MASL_SANYAL, MASH_F0_SANYAL, MASH_F1_SANYAL, MASH_F2_SANYAL, MASH_F34_SANYAL, nrow = 3, ncol = 2, align = "hv")

dev.off()













#same (similar) for UCAM
normalised_UCAM_MASL = rowMeans(subset(normalised_beforecorrection, select = which(codes$Dataset == "UCAM" & codes$SAF.score == "MASL"), na.rm = TRUE))
corrected_UCAM_MASL = rowMeans(subset(corrected, select = which(codes$Dataset == "UCAM" & codes$SAF.score == "MASL"), na.rm = TRUE))

normalised_UCAM_MASH_F0 = rowMeans(subset(normalised_beforecorrection, select = which(codes$Dataset == "UCAM" & codes$SAF.score == "MASH F0"), na.rm = TRUE))
corrected_UCAM_MASH_F0 = rowMeans(subset(corrected, select = which(codes$Dataset == "UCAM" & codes$SAF.score == "MASH F0"), na.rm = TRUE))

normalised_UCAM_MASH_F1 = rowMeans(subset(normalised_beforecorrection, select = which(codes$Dataset == "UCAM" & codes$SAF.score == "MASH F1"), na.rm = TRUE))
corrected_UCAM_MASH_F1 = rowMeans(subset(corrected, select = which(codes$Dataset == "UCAM" & codes$SAF.score == "MASH F1"), na.rm = TRUE))

normalised_UCAM_MASH_F2 = rowMeans(subset(normalised_beforecorrection, select = which(codes$Dataset == "UCAM" & codes$SAF.score == "MASH F2"), na.rm = TRUE))
corrected_UCAM_MASH_F2 = rowMeans(subset(corrected, select = which(codes$Dataset == "UCAM" & codes$SAF.score == "MASH F2"), na.rm = TRUE))

normalised_UCAM_MASH_F3 = rowMeans(subset(normalised_beforecorrection, select = which(codes$Dataset == "UCAM" & codes$SAF.score == "MASH F3"), na.rm = TRUE))
corrected_UCAM_MASH_F3 = rowMeans(subset(corrected, select = which(codes$Dataset == "UCAM" & codes$SAF.score == "MASH F3"), na.rm = TRUE))

normalised_UCAM_MASH_F4 = rowMeans(subset(normalised_beforecorrection, select = which(codes$Dataset == "UCAM" & codes$SAF.score == "MASH F4"), na.rm = TRUE))
corrected_UCAM_MASH_F4 = rowMeans(subset(corrected, select = which(codes$Dataset == "UCAM" & codes$SAF.score == "MASH F4"), na.rm = TRUE))



#plot meanNAS in each SW
library("ggpubr")
library(cowplot)
pdf("before-after correction comparison UCAM.pdf")

mydata = data.frame("normalised_UCAM_MASL" = normalised_UCAM_MASL, "corrected_UCAM_MASL" = corrected_UCAM_MASL)
MASL_UCAM <- ggscatter(mydata, x = "normalised_UCAM_MASL", y = "corrected_UCAM_MASL",
                         color = "#f3acac", shape = 20, size = 1, alpha = 0.3, # Points color, shape, size and transparency
                         add = "reg.line",  # Add regressin line
                         add.params = list(color = "black", fill = "lightgray"), conf.int = TRUE, 
                         cor.coef = TRUE, cor.method = "pearson",
                         #ylim = c(1,6),
                         xlab = "Normalised MASL\n(UCAM)", ylab = "Corrected MASL\n(UCAM)", main = "MASL UCAM")

mydata = data.frame("normalised_UCAM_MASH_F0" = normalised_UCAM_MASH_F0, "corrected_UCAM_MASH_F0" = corrected_UCAM_MASH_F0)

mydata = data.frame("normalised_UCAM_MASH_F1" = normalised_UCAM_MASH_F1, "corrected_UCAM_MASH_F1" = corrected_UCAM_MASH_F1)
MASH_F1_UCAM <- ggscatter(mydata, x = "normalised_UCAM_MASH_F1", y = "corrected_UCAM_MASH_F1",
                            color = "#de1e1d", shape = 20, size = 1, alpha = 0.3, # Points color, shape, size and transparency
                            add = "reg.line",  # Add regressin line
                            add.params = list(color = "black", fill = "lightgray"), conf.int = TRUE, 
                            cor.coef = TRUE, cor.method = "pearson",
                            #ylim = c(1,6),
                            xlab = "Normalised MASHF1\n(UCAM)", ylab = "Corrected MASHF1\n(UCAM)", main = "MASHF1 UCAM")

mydata = data.frame("normalised_UCAM_MASH_F2" = normalised_UCAM_MASH_F2, "corrected_UCAM_MASH_F2" = corrected_UCAM_MASH_F2)
MASH_F2_UCAM <- ggscatter(mydata, x = "normalised_UCAM_MASH_F2", y = "corrected_UCAM_MASH_F2",
                            color = "#971414", shape = 20, size = 1, alpha = 0.3, # Points color, shape, size and transparency
                            add = "reg.line",  # Add regressin line
                            add.params = list(color = "black", fill = "lightgray"), conf.int = TRUE, 
                            cor.coef = TRUE, cor.method = "pearson",
                            #ylim = c(1,6),
                            xlab = "Normalised MASHF2\n(UCAM)", ylab = "Corrected MASHF2\n(UCAM)", main = "MASHF2 UCAM")

mydata = data.frame("normalised_UCAM_MASH_F3" = normalised_UCAM_MASH_F3, "corrected_UCAM_MASH_F3" = corrected_UCAM_MASH_F3)
MASH_F3_UCAM <- ggscatter(mydata, x = "normalised_UCAM_MASH_F3", y = "corrected_UCAM_MASH_F3",
                             color = "#4f0a0a", shape = 20, size = 1, alpha = 0.3, # Points color, shape, size and transparency
                             add = "reg.line",  # Add regressin line
                             add.params = list(color = "black", fill = "lightgray"), conf.int = TRUE, 
                             cor.coef = TRUE, cor.method = "pearson",
                             #ylim = c(1,6),
                             xlab = "Normalised MASHF3\n(UCAM)", ylab = "Corrected MASHF3\n(UCAM)", main = "MASHF3 UCAM")

mydata = data.frame("normalised_UCAM_MASH_F4" = normalised_UCAM_MASH_F4, "corrected_UCAM_MASH_F4" = corrected_UCAM_MASH_F4)
MASH_F4_UCAM <- ggscatter(mydata, x = "normalised_UCAM_MASH_F4", y = "corrected_UCAM_MASH_F4",
                          color = "#400808", shape = 20, size = 1, alpha = 0.3, # Points color, shape, size and transparency
                          add = "reg.line",  # Add regressin line
                          add.params = list(color = "black", fill = "lightgray"), conf.int = TRUE, 
                          cor.coef = TRUE, cor.method = "pearson",
                          #ylim = c(1,6),
                          xlab = "Normalised MASHF4\n(UCAM)", ylab = "Corrected MASHF4\n(UCAM)", main = "MASHF4 UCAM")

plot_grid(MASL_UCAM, MASH_F1_UCAM, MASH_F2_UCAM, MASH_F3_UCAM, MASH_F4_UCAM, nrow = 3, ncol = 2, align = "hv")

dev.off()












# --- Stratified correlation analysis ---
library(ggplot2)
library(dplyr)

# Compute mean expression across all samples (before correction)
mean_expr <- rowMeans(normalised_beforecorrection, na.rm = TRUE)

# Divide genes into three expression bins (low, medium, high)
expr_bin <- cut(mean_expr,
                breaks = quantile(mean_expr, probs = c(0, 1/3, 2/3, 1)),
                labels = c("Low", "Medium", "High"),
                include.lowest = TRUE)

# Compute per-gene correlation between before and after correction
# Using fast vectorized row-wise correlation
rowCor <- function(x, y) {
  xm <- rowMeans(x)
  ym <- rowMeans(y)
  num <- rowSums((x - xm) * (y - ym))
  den <- sqrt(rowSums((x - xm)^2) * rowSums((y - ym)^2))
  num / den
}

cor_values <- rowCor(as.matrix(normalised_beforecorrection), as.matrix(corrected))

# Combine results
df_cor <- data.frame(
  gene = names(mean_expr),
  mean_expr = mean_expr,
  expr_bin = expr_bin,
  cor = cor_values
)

# Summarise mean correlation per expression bin
summary_df <- df_cor %>%
  group_by(expr_bin) %>%
  summarise(mean_correlation = mean(cor, na.rm = TRUE))

# Plot
pdf("Expression_Stratified_Correlation.pdf", width = 2.5, height = 2)
ggplot(summary_df, aes(x = expr_bin, y = mean_correlation, fill = expr_bin)) +
  geom_bar(stat = "identity", color = "black", width = 0.6) +
  scale_fill_manual(values = c("#B8E186", "#4DAC26", "#0571B0")) +
  labs(x = "Expression level", y = "Mean Pearson correlation") +
  coord_cartesian(ylim = c(0, 1)) +   # <-- y-axis from 0.4 to 1
  theme_minimal(base_size = 9) +
  theme(
    legend.position = "none",
    axis.text = element_text(size = 8),
    axis.title = element_text(size = 9)
  )
dev.off()


# Print numeric summary
print(summary_df)

















# --- Stratified correlation per disease stage ---
library(dplyr)
library(ggplot2)

# Define a helper function to compute row-wise Pearson correlations quickly
rowCor <- function(x, y) {
  xm <- rowMeans(x)
  ym <- rowMeans(y)
  num <- rowSums((x - xm) * (y - ym))
  den <- sqrt(rowSums((x - xm)^2) * rowSums((y - ym)^2))
  num / den
}

# Prepare metadata
codes <- codes %>%
  mutate(SAF.score = factor(SAF.score, levels = c("CTRL", "MASL", "MASH F0", "MASH F1", "MASH F2", "MASH F3", "MASH F4")))

# Prepare expression bins (based on all samples)
mean_expr <- rowMeans(normalised_beforecorrection, na.rm = TRUE)
expr_bin <- cut(mean_expr,
                breaks = quantile(mean_expr, probs = c(0, 1/3, 2/3, 1)),
                labels = c("Low", "Medium", "High"),
                include.lowest = TRUE)

# Create list of disease stages
stages <- levels(codes$SAF.score)
stages <- stages[!is.na(stages)]

# Initialize results storage
results_list <- list()

# Loop through disease stages
for (stage in stages) {
  samples_stage <- codes$Sample.name[codes$SAF.score == stage]
  
  if (length(samples_stage) < 3) next  # skip if not enough samples
  
  expr_pre_stage <- normalised_beforecorrection[, colnames(normalised_beforecorrection) %in% samples_stage, drop = FALSE]
  expr_post_stage <- corrected[, colnames(corrected) %in% samples_stage, drop = FALSE]
  
  cor_stage <- rowCor(as.matrix(expr_pre_stage), as.matrix(expr_post_stage))
  
  results_list[[stage]] <- data.frame(
    gene = rownames(expr_pre_stage),
    expr_bin = expr_bin,
    cor = cor_stage,
    stage = stage
  )
}

# Combine all into one data frame
df_stage_cor <- do.call(rbind, results_list)

# Compute mean correlation per bin and stage
summary_stage_df <- df_stage_cor %>%
  group_by(stage, expr_bin) %>%
  summarise(mean_correlation = mean(cor, na.rm = TRUE), .groups = "drop")

# --- Plot grouped barplot ---
pdf("Expression_Stratified_Correlation_by_Stage.pdf", width = 4.5, height = 3)
ggplot(summary_stage_df, aes(x = expr_bin, y = mean_correlation, fill = expr_bin)) +
  geom_bar(stat = "identity", color = "black", width = 0.6) +
  facet_wrap(~stage, nrow = 2) +
  scale_fill_manual(values = c("#B8E186", "#4DAC26", "#0571B0")) +
  labs(x = "Expression level", y = "Mean Pearson correlation") +
  coord_cartesian(ylim = c(0.4, 1)) +
  theme_minimal(base_size = 9) +
  theme(
    legend.position = "none",
    axis.text = element_text(size = 8),
    axis.title = element_text(size = 9),
    strip.text = element_text(size = 8, face = "bold")
  )
dev.off()

# Print numeric summary
print(summary_stage_df)




# --- Overall correlation distribution across all stages ---
pdf("Expression_Stratified_Correlation_AllStages.pdf", width = 2.5, height = 2)

ggplot(df_stage_cor, aes(x = expr_bin, y = cor, fill = expr_bin)) +
  geom_boxplot(outlier.size = 0.5, width = 0.6, color = "black") +
  scale_fill_manual(values = c("#B8E186", "#4DAC26", "#0571B0")) +
  labs(x = "Expression level", y = "Pearson correlation") +
  coord_cartesian(ylim = c(0, 1)) +
  theme_minimal(base_size = 9) +
  theme(
    legend.position = "none",
    axis.text = element_text(size = 8),
    axis.title = element_text(size = 9)
  )

dev.off()






library(ggplot2)
library(dplyr)

# --- Compute summary statistics ---
# df_stage_cor must contain: expr_bin, stage, cor
summary_stage_df <- df_stage_cor %>%
  group_by(expr_bin, stage) %>%
  summarise(mean_correlation = mean(cor, na.rm = TRUE)) %>%
  ungroup()

# overall mean per expr_bin (for the bars)
overall_means <- summary_stage_df %>%
  group_by(expr_bin) %>%
  summarise(overall_mean = mean(mean_correlation))

# --- Barplot with stage means as dots ---
pdf("Expression_Stratified_Correlation_AllStages.pdf", width = 2.5, height = 2)

ggplot() +
  # Bars = overall mean correlation per expression bin
  geom_bar(data = overall_means, aes(x = expr_bin, y = overall_mean, fill = expr_bin),
           stat = "identity", color = "black", width = 0.6) +
  # Points = mean per disease stage within each expression bin
  geom_point(data = summary_stage_df, 
             aes(x = expr_bin, y = mean_correlation, color = stage),
             position = position_jitter(width = 0.05, height = 0), 
             size = .3) +
  scale_fill_manual(values = c("#B8E186", "#4DAC26", "#0571B0")) +
  scale_color_manual(values = c(
    "CTRL" = "black",
    "MASL" = "#f3acac",
    "MASH F0" = "#ea6564",
    "MASH F1" = "#de1e1d",
    "MASH F2" = "#971414",
    "MASH F3" = "#4f0a0a",
    "MASH F4" = "#4f0a0a"
  )) +
  labs(x = "Expression level", y = "Mean Pearson correlation") +
  coord_cartesian(ylim = c(0.4, 1)) +
  theme_minimal(base_size = 9) +
  theme(
    legend.position = "none",
    axis.text = element_text(size = 8),
    axis.title = element_text(size = 9)
  )

dev.off()











# Add standard deviation for the error bars
overall_means <- summary_stage_df %>%
  group_by(expr_bin) %>%
  summarise(
    overall_mean = mean(mean_correlation),
    sd = sd(mean_correlation),
    se = sd / sqrt(n())
  )

pdf("Expression_Stratified_Correlation_AllStages.pdf", width = 2.5, height = 2)

ggplot() +
  # Bars = overall mean correlation per expression bin
  geom_bar(data = overall_means, aes(x = expr_bin, y = overall_mean, fill = expr_bin),
           stat = "identity", color = "black", width = 0.6) +
  # Error bars showing variability
  geom_errorbar(data = overall_means, 
                aes(x = expr_bin, ymin = overall_mean - sd, ymax = overall_mean + sd),
                width = 0.15, color = "black", linewidth = 0.3) +
  # Dots per disease stage
  geom_point(data = summary_stage_df, 
             aes(x = expr_bin, y = mean_correlation, color = stage),
             position = position_jitter(width = 0.05, height = 0), 
             size = 0.3) +
  scale_fill_manual(values = c("#B8E186", "#4DAC26", "#0571B0")) +
  scale_color_manual(values = c(
    "CTRL" = "black",
    "MASL" = "#f3acac",
    "MASH F0" = "#ea6564",
    "MASH F1" = "#de1e1d",
    "MASH F2" = "#971414",
    "MASH F3" = "#4f0a0a",
    "MASH F4" = "#4f0a0a"
  )) +
  labs(x = "Expression level", y = "Mean Pearson correlation") +
  coord_cartesian(ylim = c(0.4, 1)) +
  theme_minimal(base_size = 8) +
  theme(
    legend.position = "none",
    axis.text = element_text(size = 10),
    axis.title = element_text(size = 10)
  )

dev.off()

